# Test final: baseline, shared y variantes comprimidas

Evalúa **una sola vez** sobre test los modelos con la selección ya cerrada en
validación: `B0`, `S0_sel` y, para cada réplica, `S1_d32` y `S2_d32` en su mejor
época de validación. Mismo protocolo que el test del 4 de octubre: crop central
256×256, RGB en [0, 1], batch 1, sin clipping, salida final de la última etapa.

**Análisis registrado antes de correr**

- Principal: S2 − S1, promediado entre réplicas, en PSNR macro y ponderado.
- Secundarios: cada réplica por separado, S2 y S1 frente a S0_sel y frente a B0,
  y las diferencias por tarea.
- Control: B0 y S0_sel deben reproducir el test del 4 de octubre dentro de 0,01 dB.

**Uso de test.** B0 y el shared ya se evaluaron en test el 4 de octubre y ese
resultado orientó esta fase. S1 y S2 nunca se evaluaron en test y su selección
se cerró con validación. Después de este notebook no se cambian modelos ni
épocas en función de test.

Para evitar corridas accidentales, hay que poner `CONFIRM_TEST_RUN = True` en la
celda 3. Sólo inferencia: cualquier GPU sirve. Se reanuda por modelo.

In [ ]:
# 1. Entorno: fork con los módulos de compresión
import hashlib
import os
import subprocess
import sys
from pathlib import Path

# MAXIM_NOTEBOOK_SMOKE=1 sólo se usa para validar el notebook localmente en CPU
# con un modelo reducido y datos sintéticos. En Colab queda en False.
SMOKE_TEST = os.environ.get("MAXIM_NOTEBOOK_SMOKE") == "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Margen para el paso de entrenamiento, que pide ~12 GiB en un solo bloque.
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")

REPO_URL = "https://github.com/Matiata/maxim.git"
REPO_REF = "main"  # rama o commit con maxim/models/compression.py
EXPECTED_SOURCE_SHA256 = {
    "maxim/models/maxim.py": "5f8a4e005ffc5b12b5dcfbd9b2a06cab0125dbbdb97813c4b1f8ea1b2d5bf57a",
    "maxim/models/compression.py": "b460019514b05dd0b0bbe828f2d145aa12dbc443e36554b0fc735dbc84636c4c",
    "maxim/models/compat.py": "cebfaa8bcd5eaf947c3d98962a3e2266e44265b16f0836b2d5c878581abe610c"
}


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()


if SMOKE_TEST:
    REPO_DIR = Path(os.environ["MAXIM_REPO_DIR"]).resolve()
else:
    REPO_DIR = Path("/content/maxim_compresion_src")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif git("remote", "get-url", "origin").rstrip("/") != REPO_URL:
        raise RuntimeError("REPO_DIR apunta a otro repositorio.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    try:
        target = git("rev-parse", "--verify", f"origin/{REPO_REF}")
    except subprocess.CalledProcessError:
        target = REPO_REF
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", target], check=True
    )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[cuda12]==0.10.2", "flax==0.11.2", "optax==0.2.8",
            "ml-collections==0.1.1", "einops", "Pillow",
        ],
        check=True,
    )

try:
    REPO_COMMIT = git("rev-parse", "HEAD")
except Exception as error:
    REPO_COMMIT = f"unavailable: {error}"

SOURCE_SHA256 = {}
for relative, expected in EXPECTED_SOURCE_SHA256.items():
    path = REPO_DIR / relative
    if not path.exists():
        raise FileNotFoundError(
            f"Falta {relative} en {REPO_REF}. Subir los cambios (git push)."
        )
    SOURCE_SHA256[relative] = hashlib.sha256(path.read_bytes()).hexdigest()
    if SOURCE_SHA256[relative] != expected:
        raise RuntimeError(
            f"{relative} no coincide con la versión revisada: "
            f"{SOURCE_SHA256[relative]} != {expected}."
        )

sys.path.insert(0, str(REPO_DIR))
print("Commit:", REPO_COMMIT, "| modo smoke:", SMOKE_TEST)
print("Si JAX ya estaba importado antes de instalar, reiniciar el runtime.")

In [ ]:
# 2. Drive, dependencias y GPU
if not SMOKE_TEST:
    from google.colab import drive

    drive.mount("/content/gdrive")

import collections
import gc
import importlib.metadata
import json
import platform
import re
import shutil
import time
import traceback
from collections.abc import Mapping

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import traverse_util
from flax.core import unfreeze
from flax.training import checkpoints
from PIL import Image

from maxim.evaluation_metrics import (
    aggregate_task_psnr,
    aggregate_task_ssim,
    build_best_metric_payload,
)
from maxim.models.compression import (
    VARIANTS,
    build_variant,
    flatten_params,
    is_new_local_mixer,
    removed_rcab_conv_at,
    transplant_params,
    variant_replaced_levels,
)
from maxim.models.maxim import RCAB

expected_versions = {"jax": "0.10.2", "flax": "0.11.2"}
actual_versions = {
    name: importlib.metadata.version(name) for name in expected_versions
}
if actual_versions != expected_versions:
    raise RuntimeError(f"Versiones inesperadas: {actual_versions}; reiniciar.")
GPU_DEVICES = [device for device in jax.devices() if device.platform == "gpu"]
if not GPU_DEVICES and not SMOKE_TEST:
    raise RuntimeError("Seleccionar un runtime GPU en Colab.")
DEVICE = GPU_DEVICES[0] if GPU_DEVICES else jax.devices()[0]
DEVICE_KIND = getattr(DEVICE, "device_kind", DEVICE.platform)
print("Dispositivo:", DEVICE, DEVICE_KIND)

In [ ]:
# 3. Configuración
SPLIT = "test"
TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]
if SMOKE_TEST:
    DATA_ROOT = Path(os.environ["MAXIM_SMOKE_DATA"])
    CHECKPOINT_ROOT = Path(os.environ["MAXIM_SMOKE_ROOT"])
    LOCAL_CACHE = Path(os.environ["MAXIM_SMOKE_ROOT"]) / "cache"
else:
    DATA_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier")
    CHECKPOINT_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/ckpts")
    LOCAL_CACHE = Path("/content/cache_compresion")
OUTPUT_DIR = CHECKPOINT_ROOT / "compresion_test_final"
RESULT_PATH = OUTPUT_DIR / "test_final_compresion_results.json"
REPORT_PATH = OUTPUT_DIR / "test_final_compresion.md"
BENCHMARK_PATH = (CHECKPOINT_ROOT / "benchmark_maxim_v2_compresion"
                  / "benchmark_v2_results.json")
FLOPS_PATH = (REPO_DIR / "runs_logs" / "benchmark_maxim_v2_compresion_results"
              / "flops_previos_optimizacion_cpu.json")
# Salvaguarda: el test se evalúa una sola vez, con la selección cerrada.
CONFIRM_TEST_RUN = False  # @param {type:"boolean"}

MODEL_CONFIG = {
    "variant": "S-2", "dropout_rate": 0.1, "num_outputs": 3,
    "use_bias": True, "num_supervision_scales": 3,
}
BASELINE_SPEC = {
    "relative_run": "maxim_no_moe/baseline_S-2_from_newsplit_ckpt_extra10",
    "epoch": 10, "expected_step": 20000, "val_macro": 31.874760357289738,
}
SHARED_SPEC = {
    "relative_run": "moe_shared_residual_from_newsplit_baseline_extra10",
    "epoch": 8, "expected_step": 16000, "val_macro": 32.06335479306456,
}
# Resultados del test del 4 de octubre: S0_sel calcula exactamente lo mismo
# que el shared denso evaluado entonces.
REFERENCE_MACRO = {"B0": 28.12742717709229, "S0_sel": 28.13664076320675}

# Réplicas de la adaptación. Si sólo existe v1, dejar ["v1"].
REPLICAS = ["v1", "v2"]
EVAL_MODELS = [
    {"name": "B0", "variant": "B0", "source": "reference", "spec": BASELINE_SPEC},
    {"name": "S0_sel", "variant": "S0_sel", "source": "reference",
     "spec": SHARED_SPEC},
] + [
    {"name": f"{variant}_{replica}", "variant": variant, "source": "adaptation",
     "replica": replica, "run": f"compresion_adaptacion/{variant}_{replica}"}
    for replica in REPLICAS for variant in ("S1_d32", "S2_d32")
]
BF16_MODELS = []  # bf16 ya se evaluó en la primera validación
COMPARISONS = [("S0_sel", "B0")]
for replica in REPLICAS:
    COMPARISONS += [(f"S2_d32_{replica}", f"S1_d32_{replica}"),
                    (f"S1_d32_{replica}", "S0_sel"),
                    (f"S2_d32_{replica}", "S0_sel"),
                    (f"S2_d32_{replica}", "B0")]
# Diferencias promediadas entre réplicas: el resultado principal.
SEED_AVERAGED = {
    "S2_d32 - S1_d32": [(f"S2_d32_{r}", f"S1_d32_{r}") for r in REPLICAS],
    "S2_d32 - S0_sel": [(f"S2_d32_{r}", "S0_sel") for r in REPLICAS],
    "S1_d32 - S0_sel": [(f"S1_d32_{r}", "S0_sel") for r in REPLICAS],
    "S2_d32 - B0": [(f"S2_d32_{r}", "B0") for r in REPLICAS],
}
EVAL_PATCH_SIZE = 256
BOOTSTRAP_SAMPLES = 2000
BOOTSTRAP_SEED = 20261023
REFERENCE_TOLERANCE_DB = 0.01
REQUIRE_COMPLETE_ADAPTATION = True
MODEL_VARIANT = {entry["name"]: entry["variant"] for entry in EVAL_MODELS}

if SMOKE_TEST:
    MODEL_CONFIG.update(features=8, block_size_hr=(4, 4), grid_size_hr=(4, 4),
                        block_size_lr=(2, 2), grid_size_lr=(2, 2))
    EVAL_PATCH_SIZE, BOOTSTRAP_SAMPLES = 32, 50
print("Split:", SPLIT, "| réplicas:", REPLICAS, "| resultados:", RESULT_PATH)

In [ ]:
# 4. Utilidades: JSON atómico, hashes, entorno y listas de datos
def json_safe(value):
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if isinstance(value, (float, int, str, bool)) or value is None:
        return value
    return str(value)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2,
                   allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def stable_hash(payload):
    return hashlib.sha256(
        json.dumps(json_safe(payload), sort_keys=True).encode("utf-8")
    ).hexdigest()


def hash_params(tree):
    digest = hashlib.sha256()
    for key, value in sorted(flatten_params(tree).items()):
        array = np.ascontiguousarray(np.asarray(jax.device_get(value)))
        if not np.isfinite(array).all():
            raise FloatingPointError(f"Parámetro no finito: {key}")
        digest.update(json.dumps([str(part) for part in key]).encode("utf-8"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(array.tobytes())
    return digest.hexdigest()


def environment_payload():
    packages = {}
    for name in ("jax", "jaxlib", "flax", "optax", "orbax-checkpoint",
                 "numpy", "tensorflow"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": str(DEVICE),
        "device_kind": DEVICE_KIND,
        "repo_commit": REPO_COMMIT,
        "source_sha256": SOURCE_SHA256,
        "mem_fraction": os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"),
        "smoke_test": SMOKE_TEST,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


def read_split_names(task, split):
    path = DATA_ROOT / task / f"{split}.txt"
    raw = path.read_bytes()
    names = [line.strip() for line in raw.decode("utf-8").splitlines()
             if line.strip()]
    if len(names) != len(set(names)):
        raise ValueError(f"Entradas duplicadas en {path}")
    for name in names:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Ruta inválida en {path}: {name}")
    return names, {"count": len(names),
                   "sha256": hashlib.sha256(raw).hexdigest()}


def verify_split_manifest(reference_run_metadata):
    """Las listas actuales deben ser idénticas a las de la run shared fuente."""
    reference = reference_run_metadata["split_manifest"]
    manifest = {}
    for task in TASKS:
        manifest[task] = {}
        for split in ("train", "val", "test"):
            _, value = read_split_names(task, split)
            wanted = {key: reference[task][split][key]
                      for key in ("count", "sha256")}
            if value != wanted and not SMOKE_TEST:
                raise ValueError(
                    f"Lista distinta de la run fuente: {task}/{split}: "
                    f"{value} != {wanted}")
            manifest[task][split] = value
    return manifest

In [ ]:
# Métricas idénticas a las de los notebooks de entrenamiento y de test
SSIM_DATA_RANGE, SSIM_FILTER_SIZE, SSIM_FILTER_SIGMA = 1.0, 11, 1.5
SSIM_K1, SSIM_K2 = 0.01, 0.03


def compute_psnr_per_example(pred, target):
    diff = pred * 255.0 - target * 255.0
    mse = jnp.mean(diff ** 2, axis=(1, 2, 3))
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))


def compute_ssim_per_example(pred, target):
    """Single-scale RGB SSIM matching tf.image.ssim defaults."""
    pred = jnp.asarray(pred, dtype=jnp.float32)
    target = jnp.asarray(target, dtype=jnp.float32)
    if pred.ndim != 4 or pred.shape != target.shape:
        raise ValueError(f"SSIM expects equal NHWC tensors, got {pred.shape} "
                         f"and {target.shape}.")
    coordinates = jnp.arange(SSIM_FILTER_SIZE, dtype=jnp.float32)
    coordinates -= (SSIM_FILTER_SIZE - 1.0) / 2.0
    gaussian_1d = jnp.exp(-0.5 * (coordinates / SSIM_FILTER_SIGMA) ** 2)
    gaussian_1d /= jnp.sum(gaussian_1d)
    gaussian_2d = gaussian_1d[:, None] * gaussian_1d[None, :]
    channels = pred.shape[-1]
    kernel = jnp.tile(gaussian_2d[:, :, None, None], (1, 1, 1, channels))

    def depthwise_filter(images):
        return jax.lax.conv_general_dilated(
            images, kernel, window_strides=(1, 1), padding="VALID",
            dimension_numbers=("NHWC", "HWIO", "NHWC"),
            feature_group_count=channels,
        )

    mean_pred = depthwise_filter(pred)
    mean_target = depthwise_filter(target)
    variance_pred = depthwise_filter(pred * pred) - mean_pred * mean_pred
    variance_target = (depthwise_filter(target * target)
                       - mean_target * mean_target)
    covariance = depthwise_filter(pred * target) - mean_pred * mean_target
    c1 = (SSIM_K1 * SSIM_DATA_RANGE) ** 2
    c2 = (SSIM_K2 * SSIM_DATA_RANGE) ** 2
    luminance = (2.0 * mean_pred * mean_target + c1) / (
        mean_pred ** 2 + mean_target ** 2 + c1)
    contrast_structure = (2.0 * covariance + c2) / (
        variance_pred + variance_target + c2)
    return jnp.mean(luminance * contrast_structure, axis=(1, 2, 3))


_probe = np.linspace(0, 1, 2 * 16 * 16 * 3, dtype=np.float32).reshape(2, 16, 16, 3)
np.testing.assert_allclose(
    np.asarray(jax.jit(compute_ssim_per_example)(jnp.asarray(_probe),
                                                 jnp.asarray(_probe))),
    1.0, atol=1e-5)
print("Métricas sintéticas OK")

In [ ]:
# Validación en memoria: crop central 256×256, igual que en entrenamiento y test
def center_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect")
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect")
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)
    return (image[top:top + crop_size, left:left + crop_size],
            target[top:top + crop_size, left:left + crop_size])


def load_uint8(path, retries=5):
    last_error = None
    for attempt in range(retries):
        try:
            with Image.open(path) as image:
                return np.asarray(image.convert("RGB"), dtype=np.uint8)
        except OSError as error:
            last_error = error
            time.sleep(0.3)
    raise OSError(f"No se pudo leer {path}") from last_error


def build_val_cache(split="val"):
    """Crops centrales en uint8. Dividir por 255 da los mismos valores float32
    que el pipeline original, que convertía a float antes de recortar."""
    names, inputs, targets, task_ids = [], [], [], []
    for task_id, task in enumerate(TASKS):
        task_names, _ = read_split_names(task, split)
        for name in task_names:
            x = load_uint8(DATA_ROOT / task / "imgs" / name)
            y = load_uint8(DATA_ROOT / task / "GT" / name)
            x, y = center_crop(x, y, EVAL_PATCH_SIZE)
            if x.shape != (EVAL_PATCH_SIZE, EVAL_PATCH_SIZE, 3) or y.shape != x.shape:
                raise ValueError(f"Shape inválida en {task}/{name}: {x.shape}")
            inputs.append(x)
            targets.append(y)
            task_ids.append(task_id)
            names.append(f"{task}/{name}")
        print(f"{split} en memoria: {task}, {len(task_names)} pares")
    return {
        "x": np.stack(inputs), "y": np.stack(targets),
        "task_id": np.asarray(task_ids, dtype=np.int32), "names": names,
    }


def load_or_build_val_cache(cache_path, manifest_hash, split="val"):
    cache_path = Path(cache_path)
    if cache_path.exists():
        data = np.load(cache_path, allow_pickle=False)
        if str(data["manifest_hash"]) == manifest_hash:
            print("Validación en memoria reutilizada:", cache_path)
            return {"x": data["x"], "y": data["y"], "task_id": data["task_id"],
                    "names": list(data["names"])}
    cache = build_val_cache(split)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache_path, manifest_hash=manifest_hash, x=cache["x"],
             y=cache["y"], task_id=cache["task_id"],
             names=np.asarray(cache["names"]))
    return cache


def summarize_per_image(psnr, ssim, task_ids):
    psnr, ssim = np.asarray(psnr, np.float64), np.asarray(ssim, np.float64)
    task_ids = np.asarray(task_ids)
    counts = [int(np.sum(task_ids == t)) for t in range(len(TASKS))]
    psnr_sum = [float(np.sum(psnr[task_ids == t])) for t in range(len(TASKS))]
    ssim_sum = [float(np.sum(ssim[task_ids == t])) for t in range(len(TASKS))]
    p = aggregate_task_psnr(psnr_sum, counts)
    s = aggregate_task_ssim(ssim_sum, counts)
    return {
        "psnr_macro": float(p["psnr_macro"]),
        "psnr_weighted": float(p["psnr_weighted"]),
        "ssim_macro": float(s["ssim_macro"]),
        "ssim_weighted": float(s["ssim_weighted"]),
        "task_psnr": [float(v) for v in p["task_psnr"]],
        "task_ssim": [float(v) for v in s["task_ssim"]],
        "task_count": counts,
        "sample_count": int(sum(counts)),
    }

In [ ]:
# Restauración estricta de checkpoints de referencia
def variant_family(name):
    return VARIANTS[name.split("@")[0]]["family"]


def template_params(model, name):
    kwargs = ({} if variant_family(name) == "baseline"
              else {"task_id": jnp.zeros((1,), dtype=jnp.int32)})
    dummy = jnp.zeros((1, 64, 64, 3), dtype=jnp.float32)
    if SMOKE_TEST:
        dummy = jnp.zeros((1, 32, 32, 3), dtype=jnp.float32)
    return jax.eval_shape(lambda: model.init(
        jax.random.PRNGKey(0), dummy, train=False, **kwargs))["params"]


def validate_tree(expected, actual, label):
    wanted, found = flatten_params(expected), flatten_params(actual)
    missing = sorted(set(wanted) - set(found))
    extra = sorted(set(found) - set(wanted))
    if missing or extra:
        raise ValueError(f"{label}: faltan {missing[:4]}, sobran {extra[:4]}")
    for key in wanted:
        if (tuple(wanted[key].shape) != tuple(found[key].shape)
                or np.dtype(wanted[key].dtype) != np.dtype(found[key].dtype)):
            raise ValueError(f"{label}: shape o dtype incompatible en {key}")


def restore_reference_params(spec, template):
    run_dir = CHECKPOINT_ROOT / spec["relative_run"]
    selection = read_json(run_dir / "best_checkpoint" / "best_metric.json")
    if selection.get("selection_metric") != "psnr_macro":
        raise ValueError(f"{spec['relative_run']}: selección no macro.")
    if int(selection["epoch"]) != spec["epoch"]:
        raise ValueError(f"{spec['relative_run']}: cambió la época elegida.")
    if abs(float(selection["psnr_macro"]) - spec["val_macro"]) > 1e-6:
        raise ValueError(f"{spec['relative_run']}: cambió el PSNR elegido.")
    raw = checkpoints.restore_checkpoint(
        ckpt_dir=str(run_dir / "best_checkpoint" / f"checkpoint_{spec['epoch']}"),
        target=None)
    if not isinstance(raw, Mapping) or "params" not in raw or "step" not in raw:
        raise ValueError("No se restauró un TrainState completo.")
    if int(np.asarray(raw["step"])) != spec["expected_step"]:
        raise ValueError(f"Paso inesperado en {spec['relative_run']}.")
    validate_tree(template, raw["params"], spec["relative_run"])
    params = jax.tree_util.tree_map(jnp.asarray, raw["params"])
    return params, {
        "checkpoint": str(run_dir / "best_checkpoint" / f"checkpoint_{spec['epoch']}"),
        "restored_step": int(np.asarray(raw["step"])),
        "params_sha256": hash_params(params),
        "selection": selection,
        "run_metadata": read_json(run_dir / "run_metadata.json"),
    }



def load_adaptation_params(entry, template):
    run_dir = CHECKPOINT_ROOT / entry["run"]
    metadata = read_json(run_dir / "run_metadata.json")
    if metadata["variant"] != entry["variant"]:
        raise ValueError(f"{entry['run']} es de la variante {metadata['variant']}.")
    total_epochs = sum(p["epochs"] for p in metadata["protocol"]["phases"])
    completed = metadata.get("completed_epochs", 0)
    if REQUIRE_COMPLETE_ADAPTATION and completed != total_epochs:
        raise ValueError(f"{entry['run']}: {completed}/{total_epochs} épocas.")
    selection = read_json(run_dir / "best_checkpoint" / "best_metric.json")
    if selection.get("selection_metric") != "psnr_macro":
        raise ValueError(f"{entry['run']}: selección no macro.")
    epoch = int(selection["epoch"])
    raw = checkpoints.restore_checkpoint(
        ckpt_dir=str(run_dir / "best_checkpoint" / f"checkpoint_{epoch}"),
        target=None)
    if int(np.asarray(raw["epoch"])) != epoch:
        raise ValueError(f"{entry['run']}: checkpoint y metadata no coinciden.")
    validate_tree(template, raw["params"], entry["run"])
    params = jax.tree_util.tree_map(jnp.asarray, raw["params"])
    return params, {
        "checkpoint": str(run_dir / "best_checkpoint" / f"checkpoint_{epoch}"),
        "selected_epoch": epoch,
        "completed_epochs": completed,
        "total_epochs": total_epochs,
        "params_sha256": hash_params(params),
        "selection": selection,
        "protocol_hash": metadata["protocol_hash"],
        "protocol": metadata["protocol"],
        "teacher_params_sha256": metadata["teacher_source"]["params_sha256"],
        "device_kinds": sorted({s["device_kind"] for s in metadata["sessions"]}),
    }

In [ ]:
# 5. Datos en memoria, resultado reanudable y verificación de procedencia
if not CONFIRM_TEST_RUN and not SMOKE_TEST:
    raise RuntimeError("Poner CONFIRM_TEST_RUN = True para evaluar test. "
                       "Hacerlo sólo con la selección cerrada.")
SHARED_RUN_METADATA = read_json(
    CHECKPOINT_ROOT / SHARED_SPEC["relative_run"] / "run_metadata.json")
SPLIT_MANIFEST = verify_split_manifest(SHARED_RUN_METADATA)
SPLIT_DATA = load_or_build_val_cache(
    LOCAL_CACHE / f"{SPLIT}_{EVAL_PATCH_SIZE}.npz",
    stable_hash({t: SPLIT_MANIFEST[t][SPLIT] for t in TASKS}), SPLIT)

PROTOCOL = {
    "split": SPLIT,
    "split_manifest": {t: SPLIT_MANIFEST[t][SPLIT] for t in TASKS},
    "replicas": REPLICAS,
    "reference_macro": REFERENCE_MACRO,
    "primary_analysis": "S2_d32 - S1_d32 promediado entre réplicas, PSNR macro y ponderado",
    "eval_patch_size": EVAL_PATCH_SIZE,
    "models": EVAL_MODELS,
    "bf16_models": BF16_MODELS,
    "model_config": MODEL_CONFIG,
    "prediction": "última etapa, última escala; train=False; sin clipping",
    "psnr": "media por imagen; MSE en escala 0..255, piso 1e-6",
    "ssim": "RGB, gaussiana 11, sigma 1,5, VALID, rango 1",
}
PROTOCOL_HASH = stable_hash(PROTOCOL)
if RESULT_PATH.exists():
    REPORT = read_json(RESULT_PATH)
    if REPORT["protocol_hash"] != PROTOCOL_HASH:
        raise RuntimeError("Existe un resultado de otro protocolo; cambiar OUTPUT_DIR.")
else:
    REPORT = {"status": "in_progress", "protocol_hash": PROTOCOL_HASH,
              "protocol": PROTOCOL, "models": {}, "sources": {}}
REPORT.setdefault("sessions", []).append(environment_payload())
atomic_json(RESULT_PATH, REPORT)

MODELS, PARAMS, SOURCES = {}, {}, {}
for entry in EVAL_MODELS:
    model = build_variant(entry["variant"], MODEL_CONFIG, num_tasks=len(TASKS))
    template = template_params(model, entry["variant"])
    if entry["source"] == "reference":
        params, source = restore_reference_params(entry["spec"], template)
        source.pop("run_metadata")
    else:
        params, source = load_adaptation_params(entry, template)
    MODELS[entry["name"]], PARAMS[entry["name"]] = model, params
    SOURCES[entry["name"]] = source
    previous = REPORT["sources"].get(entry["name"])
    if previous and previous["params_sha256"] != source["params_sha256"]:
        raise RuntimeError(f"Cambió el checkpoint de {entry['name']}; no mezclar.")
    REPORT["sources"][entry["name"]] = source
    print(f"{entry['name']}: {sum(np.prod(v.shape) for v in jax.tree_util.tree_leaves(params)):,} parámetros")

adapted = [e for e in EVAL_MODELS if e["source"] == "adaptation"]


def protocol_without_seeds(name):
    protocol = dict(SOURCES[name]["protocol"])
    protocol.pop("seed", None)
    protocol.pop("new_module_seed", None)
    return stable_hash(protocol)


seeds = {e["replica"]: (SOURCES[e["name"]]["protocol"]["seed"],
                        SOURCES[e["name"]]["protocol"]["new_module_seed"])
         for e in adapted}
REPORT["checks"] = {
    "same_protocol_within_replica": all(
        len({SOURCES[e["name"]]["protocol_hash"] for e in adapted
             if e["replica"] == r}) == 1 for r in REPLICAS),
    "same_protocol_except_seeds_across_replicas": len(
        {protocol_without_seeds(e["name"]) for e in adapted}) == 1,
    "distinct_seeds_per_replica": len(set(seeds.values())) == len(REPLICAS),
    "replica_seeds": seeds,
    "adaptation_teacher_is_shared_checkpoint": {
        SOURCES[e["name"]]["teacher_params_sha256"] for e in adapted} == {
        SOURCES["S0_sel"]["params_sha256"]},
    "adaptation_device_kinds": {e["name"]: SOURCES[e["name"]]["device_kinds"]
                                for e in adapted},
    "selected_epochs": {e["name"]: SOURCES[e["name"]]["selected_epoch"]
                        for e in adapted},
}
for key in ("same_protocol_within_replica",
            "same_protocol_except_seeds_across_replicas",
            "distinct_seeds_per_replica",
            "adaptation_teacher_is_shared_checkpoint"):
    if not REPORT["checks"][key]:
        raise ValueError(f"Falló la verificación de procedencia: {key}")
atomic_json(RESULT_PATH, REPORT)
print(json.dumps(REPORT["checks"], indent=1))

In [ ]:
# 6. Evaluación por modelo, reanudable
def cast_floating(tree, dtype):
    return jax.tree_util.tree_map(
        lambda leaf: leaf.astype(dtype)
        if jnp.issubdtype(leaf.dtype, jnp.floating) else leaf, tree)


def make_eval_step(model, baseline, dtype):
    def step(params, x, y, task_id):
        x_in = x.astype(dtype)
        if baseline:
            preds = model.apply({"params": params}, x_in, train=False)
        else:
            preds, _ = model.apply({"params": params}, x_in, train=False,
                                   task_id=task_id)
        final = preds[-1][-1].astype(jnp.float32)
        return compute_psnr_per_example(final, y), compute_ssim_per_example(final, y)
    return jax.jit(step)


jobs = [(e["name"], jnp.float32) for e in EVAL_MODELS]
jobs += [(name, jnp.bfloat16) for name in BF16_MODELS]
for name, dtype in jobs:
    key = name if dtype == jnp.float32 else f"{name}@bf16"
    if REPORT["models"].get(key, {}).get("status") == "complete":
        print(key, "ya evaluado")
        continue
    variant = next(e["variant"] for e in EVAL_MODELS if e["name"] == name)
    step = make_eval_step(MODELS[name], variant_family(variant) == "baseline", dtype)
    params = cast_floating(PARAMS[name], dtype)
    started, psnr, ssim = time.time(), [], []
    for index in range(len(SPLIT_DATA["task_id"])):
        x = jnp.asarray(SPLIT_DATA["x"][index:index + 1].astype(np.float32) / 255.0)
        y = jnp.asarray(SPLIT_DATA["y"][index:index + 1].astype(np.float32) / 255.0)
        p, s = jax.device_get(step(params, x, y,
                                   jnp.asarray(SPLIT_DATA["task_id"][index:index + 1])))
        psnr.append(float(p[0]))
        ssim.append(float(s[0]))
    if not (np.isfinite(psnr).all() and np.isfinite(ssim).all()):
        raise FloatingPointError(f"Métrica no finita en {key}.")
    summary = summarize_per_image(psnr, ssim, SPLIT_DATA["task_id"])
    REPORT["models"][key] = {"status": "complete", "summary": summary,
                             "psnr": psnr, "ssim": ssim,
                             "seconds": time.time() - started}
    atomic_json(RESULT_PATH, REPORT)
    print(f"{key}: macro {summary['psnr_macro']:.4f} dB | ponderado "
          f"{summary['psnr_weighted']:.4f} dB | SSIM macro {summary['ssim_macro']:.4f}")
    del step, params
    gc.collect()

for name, recorded in REFERENCE_MACRO.items():
    difference = REPORT["models"][name]["summary"]["psnr_macro"] - recorded
    REPORT["checks"][f"{name}_reproduces_recorded_{SPLIT}_db"] = difference
    if abs(difference) > REFERENCE_TOLERANCE_DB and not SMOKE_TEST:
        raise AssertionError(f"{name} no reproduce su resultado registrado: "
                             f"{difference:+.4f} dB")
REPORT["status"] = "complete"
atomic_json(RESULT_PATH, REPORT)
print("Evaluación completa.")

In [ ]:
# 7. Diferencias pareadas con bootstrap estratificado por tarea
def per_image(name, metric):
    return np.asarray(REPORT["models"][name][metric], np.float64)


def stratified_bootstrap(differences):
    """IC por remuestreo de imágenes dentro de cada tarea. No incluye la
    variación entre semillas, salvo la que entra al promediar réplicas."""
    tasks = np.asarray(SPLIT_DATA["task_id"])
    groups = [differences[tasks == t] for t in range(len(TASKS))]
    observed_task = np.asarray([g.mean() for g in groups])
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = np.empty((BOOTSTRAP_SAMPLES, len(TASKS)))
    for t, g in enumerate(groups):
        idx = rng.integers(0, len(g), size=(BOOTSTRAP_SAMPLES, len(g)))
        draws[:, t] = g[idx].mean(axis=1)
    counts = np.asarray([len(g) for g in groups])

    def interval(values):
        low, high = np.percentile(values, [2.5, 97.5])
        return [float(low), float(high)]

    return {
        "macro": float(observed_task.mean()),
        "macro_ci95": interval(draws.mean(axis=1)),
        "weighted": float(differences.mean()),
        "weighted_ci95": interval((draws * counts).sum(axis=1) / counts.sum()),
        "task": observed_task.tolist(),
        "task_ci95": [interval(draws[:, t]) for t in range(len(TASKS))],
        "share_of_images_improved": float(np.mean(differences > 0)),
    }


def averaged_difference(pairs, metric):
    return np.mean([per_image(a, metric) - per_image(b, metric)
                    for a, b in pairs], axis=0)


REPORT["paired_differences"] = {}
for first, second in COMPARISONS + [(f"{n}@bf16", n) for n in BF16_MODELS]:
    if first in REPORT["models"] and second in REPORT["models"]:
        REPORT["paired_differences"][f"{first} - {second}"] = {
            metric: stratified_bootstrap(per_image(first, metric)
                                         - per_image(second, metric))
            for metric in ("psnr", "ssim")}
REPORT["seed_averaged_differences"] = {
    label: {metric: stratified_bootstrap(averaged_difference(pairs, metric))
            for metric in ("psnr", "ssim")}
    for label, pairs in SEED_AVERAGED.items()}
atomic_json(RESULT_PATH, REPORT)
print("Diferencias calculadas.")

In [ ]:
# 8. Reporte: calidad, diferencias por réplica y promediadas, costo
NOTE = "Test evaluado una sola vez con la selección cerrada en validación. El test ya se había usado el 4 de octubre para B0 y el shared; S1 y S2 no se habían evaluado en test."
def fmt(value, digits=3, signed=False):
    if value is None:
        return "—"
    return f"{value:+.{digits}f}" if signed else f"{value:.{digits}f}"


benchmark = read_json(BENCHMARK_PATH) if BENCHMARK_PATH.exists() else None
flops = read_json(FLOPS_PATH)["results"] if FLOPS_PATH.exists() else {}


def latency(variant, size):
    if not benchmark:
        return None
    m = benchmark["measurements"].get(f"inference/{size}x{size}/{variant}")
    return m["timing"]["median_ms"] if m and m.get("status") == "complete" else None


def difference_rows(differences):
    rows = []
    for label, d in differences.items():
        p, s = d["psnr"], d["ssim"]
        rows.append(
            f"| {label} | {fmt(p['macro'], 3, True)} | [{fmt(p['macro_ci95'][0], 3, True)}, "
            f"{fmt(p['macro_ci95'][1], 3, True)}] | {fmt(p['weighted'], 3, True)} | "
            f"[{fmt(p['weighted_ci95'][0], 3, True)}, {fmt(p['weighted_ci95'][1], 3, True)}]"
            f" | {fmt(s['macro'], 4, True)} | {100 * p['share_of_images_improved']:.0f} % |")
    return rows


HEADER = ("| Comparación | Δ PSNR macro | IC 95 % | Δ PSNR ponderado | IC 95 % "
          "| Δ SSIM macro | Imágenes que mejoran |")
lines = [f"# Comparación en {SPLIT}: variantes comprimidas y réplicas", "",
         f"Protocolo `{PROTOCOL_HASH}`. {len(SPLIT_DATA['task_id'])} pares, "
         f"crop central {EVAL_PATCH_SIZE}. Réplicas: {', '.join(REPLICAS)}.", "",
         "## Calidad por tarea, PSNR en dB", "",
         "| Modelo | Época | " + " | ".join(TASKS) + " | Macro | Ponderado | SSIM macro |",
         "|---|---:|" + "---:|" * (len(TASKS) + 3)]
for key, record in REPORT["models"].items():
    s = record["summary"]
    epoch = SOURCES.get(key.split("@")[0], {}).get("selected_epoch", "—")
    lines.append(f"| {key} | {epoch} | " + " | ".join(fmt(v) for v in s["task_psnr"])
                 + f" | {fmt(s['psnr_macro'], 4)} | {fmt(s['psnr_weighted'], 4)}"
                 f" | {fmt(s['ssim_macro'], 4)} |")

lines += ["", "## Diferencias promediadas entre réplicas", "",
          "Para cada imagen se promedia la diferencia de las réplicas y luego se "
          "remuestrea por tarea. Con dos réplicas, el intervalo todavía no "
          "representa bien la variación entre semillas: comparar también las "
          "réplicas por separado.", "", HEADER, "|---|---:|---|---:|---|---:|---:|"]
lines += difference_rows(REPORT["seed_averaged_differences"])
lines += ["", "## Diferencias por réplica", "", HEADER,
          "|---|---:|---|---:|---|---:|---:|"]
lines += difference_rows(REPORT["paired_differences"])

lines += ["", "## S2 − S1 por tarea", "",
          "| Tarea | " + " | ".join(f"Réplica {r}" for r in REPLICAS) + " | Media |",
          "|---|" + "---:|" * (len(REPLICAS) + 1)]
mean_task = REPORT["seed_averaged_differences"]["S2_d32 - S1_d32"]["psnr"]
for t, task in enumerate(TASKS):
    cells = []
    for r in REPLICAS:
        d = REPORT["paired_differences"][f"S2_d32_{r} - S1_d32_{r}"]["psnr"]
        cells.append(f"{fmt(d['task'][t], 3, True)} [{fmt(d['task_ci95'][t][0], 2, True)}, "
                     f"{fmt(d['task_ci95'][t][1], 2, True)}]")
    lines.append(f"| {task} | " + " | ".join(cells)
                 + f" | {fmt(mean_task['task'][t], 3, True)} |")

lines += ["", "## Calidad frente a costo", "",
          "Latencia: mediana del benchmark v2 en L4, batch 1. FLOPs: programa "
          "previo a la optimización, 256×256.", "",
          "| Modelo | Parámetros | GFLOP | ms 256 | ms 512 | PSNR macro | Δ vs S0_sel |",
          "|---|---:|---:|---:|---:|---:|---:|"]
reference = REPORT["models"]["S0_sel"]["summary"]["psnr_macro"]
for key, record in REPORT["models"].items():
    name, _, precision = key.partition("@")
    variant = MODEL_VARIANT[name]
    timed = variant + (f"@{precision}" if precision else "")
    params = int(sum(np.prod(v.shape) for v in jax.tree_util.tree_leaves(PARAMS[name])))
    gflop = flops.get(f"inference/256/{variant}", {}).get("gflop")
    macro = record["summary"]["psnr_macro"]
    lines.append(f"| {key} | {params / 1e6:.2f} M | {fmt(gflop, 1)} | "
                 f"{fmt(latency(timed, 256), 2)} | {fmt(latency(timed, 512), 1)} | "
                 f"{fmt(macro, 4)} | {fmt(macro - reference, 3, True)} |")
lines += ["", NOTE]
REPORT_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

## Entrega

Compartir `ckpts/compresion_test_final/test_final_compresion_results.json` y
`test_final_compresion.md`. Estos son los números finales para la tesis: no
volver a evaluar test con otros modelos o épocas.